   ... a fast producer (1000 Hz sensor) and a slow consumer... If you use a 
   standard dynamically growing list... will run out of memory and crash. A
   Ring Buffer (Circular Array) allocates memory exactly once. When it fills 
   up, it wraps around and overwrites the oldest, stale data with fresh data.

   ...
   - `SensorRingBuffer` class ... holds a maximum of `N` items. It must support
     `push(item)` and `readAll()`. It should silently overwrite the oldest data
     if full (Drop-Oldest policy).

     CONCEPTUAL HINTS:
     - You need an array of fixed size, a `writeIndex`, and a `count` to track
       how many items are currently valid.
     - When pushing, use the modulo operator (`%`) to wrap the `writeIndex` back
       to 0 when it hits the capacity.
     - If `count == capacity` when you push, it means you are overwriting the
       oldest data, so you must also advance the `readIndex` (or conceptually
       just know that your oldest data is now at `(writeIndex + 1) % capacity`).  


```kotlin
class SensorRingBuffer<T>(private val capacity: Int) {
    private val buffer = arrayOfNulls<Any?>(capacity) as Array<T?>
    private var writeIndex = 0
    private var readIndex = 0
    private var count = 0

    // O(1) push
    fun push(item: T) {
        buffer[writeIndex] = item
        writeIndex = (writeIndex + 1) % capacity

        if (count < capacity) { 
            count++ 
        }
        else { 
            // Drop-oldest: The oldest item was just overwritten
            // so we must advance the read index to the new "oldest" item
            readIndex = (readIndex + 1) % capacity
        }
    }

    // O(N) Read and clear
    fun readAll(): List<T> {
        val result = mutableListOf<T>()
        val curr = readIndex
        for (i in 0 until count) {
            result.add(buffer[curr]!!)
            curr = (curr + 1) % capacity
        }
        // Reset after reading
        count = 0
        readIndex = writeIndex
        return result
    }
}
```

---
2. The Deadman's Switch (State Machine / Watchdog)
   CONCEPT:
      Robots operate in the physical world. If the connection drops or the main
      compute loop freezes, the robot must not blindly execute the last command
      it receives forever. It needs a watchdog timer: a process that constantly
      checks how long it has been since the last valid heartbeat. 

   TASK: 
      Design a `SafetyMonitor` class. It receives `updateHeartbeat()` from the
      network. A separate hardware loop calls `checkSafety()`every millisecond.
      If the heartbeat is older than 500ms, the state must transition to `E_STOP`.

   CONCEPTUAL HINTS:
      - Store the timestamp of the last heartbeat.
      - Define states explicitly using an Enum.
      - In `checkSafety()`, compare the current time to the last heartbeat.

   Syntax Hints:
      - Use `enum class SystemState { ACTIVE, DEGRADED, E_STOP }`      
      - Use `System.currentTimeMillis()` to get tiemstamps.
      - Use Kotlin's `when` statement to handle transitions cleanly,.


```Kotlin
enum class SystemState {
    ACTIVE = 0,
    DEGRADED = 1,
    E_STOP = -1
}

class SafetyMonitor(private val timeoutMs: Long = 500L) {
    var currentState = SystemState.ACTIVE
        private set         // Only this class cna change the state

    private var lastHeartbeatMs: Long = System.currentTimeMillis()

    // Called by the network thread when a packet arrives
    fun updateHeartbeat() {
        if (currentState != SystemState.E_STOP) {
            lastHeartbeatMs = System.currentTimeMillis()
            currentState = SystemState.ACTIVE
        }
    }

    // Called continuously by the hardware control loop
    fun checkSafety(): SystemState {
        if (currentState == SystemState.E_STOP) {
            return currentState // Require manual reset once in E_STOP
        }

        val elapsed = System.currentTimeMillis() - lastHeartbeatMs

        if (elapsed > timeoutMs) {
            triggerEmergencyStop()
        } else if (elapsed > timeoutMs / 2) {
            // Optional: graceful degradation before full stop
            currentState = SystemState.DEGRADED
        }
    }

    private fun triggerEmergencyStop() {
        currentState = SystemState.E_STOP
        println("CRITICAL: Heartbeat lost. Motors disengaged.")
        // Hardware logic to cut power goes here
    }
}

```
         

   ... In robotics, an FSM prevents catastophic actions (like a motor trying to
   move while in an `ERROR` state). The key to acing... is explicitly defining
   STATES, EVENTS, and HANDLING INVALID TRANSITIONS.

THE CONCEPT
   A robust FSM consists of three parts:
   1. STATES: The distinct modes the robot can be in (e.g., `IDLE`, `MOVING`,
      `ERROR`).
   2. EVENTS: The triggers that attempt to change the state (e.g., `CMD_START`,
      `SENSOR_TRIPPED`).
   3. THE TRANSITION LOGIC: A strict rulebook disctating which events are allowed
      in which states.

   ... In Kotlin, you use `enum class` for States and Events, and the `when`
   expression for the transition table. Because `when` is exhaustive in Kotlin,
   it proves to the interviewer that you literally cannot forget a state.


```kotlin
// 1. Define all possible states
enum class RobotState {
    IDLE,
    MOVING,
    ERROR,
}

// 2. Define all possible triggers/events
enum class RobotEvent {
    CMD_MOVE,
    TARGET_REACHED,
    HARDWARE_FAULT,
    CMD_RESET
}

class RobotController {
    // Read-only fom the outside, only the FSM can mutate it
    var currentState: RobotState = RobotState.IDLE
        private set

    // 3. The Transition Engine
    fun handleEvent(event: RobotEvent) {
        val nextState = when (currentState) {
            RobotState.IDLE -> when (event) {
                RobotEvent.CMD_MOVE -> RobotState.MOVING
                RobotEvent.HARDWARE_FAULT -> RobotState.ERROR
                else -> handleInvalid(event)
            }

            RobotState.MOVING -> when (event) {
                RobotEvent.TARGET_REACHED -> RobotState.IDLE
                RobotEvent.HARDWARE_FAULT -> RobotState.ERROR
                else -> handleInvalid(event)
            }

            RobotState.ERROR -> when (event) {
                // In an error state, ONLY a reset can save it. Everything
                // else is ignored.
                RobotEvent.CMD_RESET -> RobotState.IDLE
                else -> handleInvalid(event)
            }
        }

        if (nextState != currentState) {
            println("Transitioning: $currentState -> $nextState on event $event")
        }
    }    
}


...



fun main() {
    val robot = RobotController()

    robot.handleEvent(RobotEvent.CMD_MOVE)
    robot.handleEvent(RobotEvent.CMD_MOVE)
    robot.handleEvent(RobotEvent.HARDWARE_FAULT)
    robot.handleEvent(RobotEvent.CMD_MOVE)
    robot.handleEvent(RobotEvent.CMD_RESET)
}
```